# Module 2: Web search with Tavily - your first agentic loop

Large language models are frozen in time — they only know what was in their training data. To answer questions about *what's happening now* (say, this week's AI news), we have to **give the model a tool** and let it go fetch fresh information.

In this notebook we build that from scratch, with no agent framework:

1. Meet the two Tavily tools we'll use: **Search** (find pages) and **Extract** (read pages in full).
2. Call Search over HTTP to see what the API looks like under the hood, then use the Python client for the rest.
3. Turn them into **tools** — plain Python functions the model can call.
4. Wire a **ReAct-style agentic loop** in ~20 lines: the model reasons, calls search/extract, reads the results, and repeats until it can answer.

By the end you'll have a tiny research agent that answers a question with real, cited sources — and you'll understand every line of it.


## 0 — Get your API keys

You need two keys, both placed in a `.env` file inside this `lesson/` folder. Copy over the `.env.example` to a new file, rename it to `.env`.

**Tavily** (web search)
1. Go to [tavily.com](https://www.tavily.com) and sign in with Google or GitHub.
2. Copy an API key from the dashboard — it looks like `tvly-...`.
3. Free tier: **1,000 credits/month, no credit card**. A basic search costs 1 credit.

**Nebius Token Factory** (the LLM)
1. Go to [tokenfactory.nebius.com](https://tokenfactory.nebius.com) and create an account.
2. Open **API keys → Create API key** and copy it (you can't view it again later).
3. Nebius exposes an **OpenAI-compatible** API, so we drive it with the plain `openai` SDK.

Your `lesson/.env` should look like:

```
TAVILY_API_KEY=tvly-...
NEBIUS_API_KEY=...
```

In [1]:
import os

from dotenv import load_dotenv
from rich import print

load_dotenv("lesson/.env")
%load_ext rich

assert os.environ.get("TAVILY_API_KEY"), "Missing TAVILY_API_KEY in lesson/.env"
assert os.environ.get("NEBIUS_API_KEY"), "Missing NEBIUS_API_KEY in lesson/.env"
print("Keys loaded.")

Keys loaded.

## The two tools we'll use today

Tavily is a search API built for LLMs — results come back as clean, structured JSON instead of raw HTML. Today we focus on two endpoints that cover most agent research workflows:

| Tool | What it does | When you use it |
|------|--------------|-----------------|
| **Tavily Search** | Given a query, finds the most relevant pages on the web and returns ranked results with titles, URLs, and content snippets. | You don't know the URL yet — you need to *discover* sources. |
| **Tavily Extract** | Given one or more URLs you already have, pulls the full page content as clean markdown/text. | You already know *where* to look — you need the page itself, not just a snippet. |

Think of them as a two-step research pattern:

1. **Search** to find promising sources.
2. **Extract** to read the ones that matter in full.

Search alone is often enough for a quick answer. Extract shines when snippets aren't enough — long docs, pricing pages, blog posts, or anything where the model needs the full text to reason accurately.

We'll peek at Search over HTTP, switch to the Python client, wrap both as tools, then let the model decide when to search vs. extract.


## 1 — Call Tavily directly over HTTP

Before any client library or agent framework, we will see what Tavily *actually is*. At its core, each capability is a single HTTP endpoint you POST to. Everything else is a wrapper around that.

We'll call **Search** by hand — enough to see the request/response shape. Extract (and everything after this section) will use the Python client.

### Search

- **Endpoint:** `POST https://api.tavily.com/search`
- **Auth:** an `Authorization: Bearer <your-key>` header
- **Body:** JSON with your `query` plus a few optional knobs

The parameters worth knowing:

| Param | What it does |
|-------|--------------|
| `query` | The search string (required) |
| `search_depth` | `basic`/`fast` (1 credit) or `advanced` (2 credits, deeper) |
| `max_results` | How many results to return (1–20) |
| `chunks_per_source` | How many snippets to return per source (1–3) |
| `include_answer` | Also return a one-shot LLM answer to the query |
| `time_range` | `day` / `week` / `month` / `year` to filter by recency |
| `include_raw_content` | Return the raw HTML content of the page |


In [2]:
import requests

response = requests.post(
    "https://api.tavily.com/search",
    headers={"Authorization": f"Bearer {os.environ['TAVILY_API_KEY']}"},
    json={
        "query": "advancements in AI this week",
        "search_depth": "advanced",
        "max_results": 5,
        "time_range": "week",
        "include_answer": True,
    },
    timeout=30,
)
response.raise_for_status()
data = response.json()

print(f"Got {len(data['results'])} results in {data['response_time']}s")
print(list(data.keys()))

Got 5 results in 3.56s

['query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id']

### What comes back

The response is a JSON object. The parts we care about:

- **`results`** — a ranked list; each item has `title`, `url`, `content` (a relevant text snippet), and a `score` (relevance, 0–1).
- **`answer`** — Tavily's own one-line answer (present only because we set `include_answer`).
- **`response_time`**, **`request_id`** — metadata.

`content` is the key field: short, relevant chunks we can feed straight to an LLM without scraping full pages.

In [3]:
print("Tavily's answer:", data.get("answer"))

print("Results: ")
for i, r in enumerate(data["results"]):
    print(f"""({i + 1}) [{r["score"]:.2f}] {r["title"]}
{r["url"]}
{r["content"][:160]}...""")

Tavily's answer: This week's AI advancements include significant developments in autonomous agents, infrastructure 
scaling, and open-weight benchmarks. Key highlights include the deployment of undercover AI agents by Apate to 
gather intelligence on fraud tactics, YouTube Music adding AI-music labels to address AI-generated music, and 
Zenzap raising $44M to compete with WhatsApp for business communication. Additionally, there are notable 
partnerships and product launches such as Google releasing a powerful new model, Workday unveiling major AI 
advancements for autonomous execution, and the US Department of Energy lending $4.4 billion to revive three nuclear
plants for AI data centers.

Results:

(1) [0.87] TechLatest AI & Tech Weekly #38
https://medium.com/@techlatest.net/techlatest-ai-tech-weekly-38-3582a39f1738
Here’s a curated roundup of our latest blogs, notable product launches, and the most interesting AI & ML updates 
from Oct 12–Oct 17, 2026.

## AI/ML News Roundu...

(2) [0.85] Google, Meta, and OpenAI make major AI updates this week
https://www.facebook.com/groups/fdmahub/posts/2077326572915968
Google released a powerful new model. Gemini Skills started rolling out. ChatGPT Voice got an upgrade. Facebook is 
letting people talk to its algorithm....

(3) [0.83] AI Is About to Change Science Forever. And This Week Gave Us a Glimpse of How. | by Erick Galaviz | 
Automation for a Calmer Life | Oct, 2026 | Medium
https://medium.com/automation-for-a-calmer-life/ai-is-about-to-change-science-forever-and-this-week-gave-us-a-glimp
se-of-how-8a21522d76a1
Press enter or click to view image in full size

What if the most important AI breakthrough of 2026 has nothing to do with writing emails, generating images, or...

(4) [0.80] News from the week beginning 12th October 2026 -
https://www.enterprisetimes.co.uk/2026/10/19/news-from-the-week-beginning-12th-october-2026
# News from the week beginning 12th October 2026

By

Steve Brooks

-

FacebookXWhatsAppLinkedinEmail

fall autumn Image by Valentin Sabau from PixabaySeveral c...

(5) [0.78] AI News Today, October 19: Top Stories | AI Weekly
https://aiweekly.co/ai-news-today
Last 4 issues — click to explore

Agents

This week: 113 Anthropic

This week: 88 Chips

This week: 84 Regulation

This week: 73 OpenAI

This week: 65 AI Infras...

## 2 — The same thing with the Python client

Writing the HTTP call by hand is great for understanding, but tedious in practice. The `tavily-python` client wraps those POST requests — same parameters, less boilerplate. This is what we'll use for the rest of the notebook.

We'll call **Search**, then **Extract** on a URL from the results — the same two-step research pattern the agent will use later.

### Search


In [5]:
%pip install tavily-python

   ---------------------------------------- 0.0/944.4 kB ? eta -:--:--
   ---------------------------------------- 944.4/944.4 kB 5.5 MB/s  0:00:00

   ------------- -------------------------- 1/3 [tiktoken]
   ------------- -------------------------- 1/3 [tiktoken]
   -------------------------- ------------- 2/3 [tavily-python]
   ---------------------------------------- 3/3 [tavily-python]

Note: you may need to restart the kernel to use updated packages.


In [6]:
from tavily import TavilyClient

# The `client_name` is optional, but it helps track attribution
tavily_client = TavilyClient(
    api_key=os.environ["TAVILY_API_KEY"], client_name="nv-course-ai-agents"
)

data = tavily_client.search(
    "advancements in AI this week",
    search_depth="advanced",
    max_results=5,
    time_range="week",
)

print(f"{len(data['results'])} results")

5 results

In [7]:
print("Results: ")
for i, r in enumerate(data["results"]):
    print(f"""({i + 1}) [{r["score"]:.2f}] {r["title"]}
{r["url"]}
{r["content"][:160]}...""")

Results:

(1) [0.87] TechLatest AI & Tech Weekly #38
https://medium.com/@techlatest.net/techlatest-ai-tech-weekly-38-3582a39f1738
Here’s a curated roundup of our latest blogs, notable product launches, and the most interesting AI & ML updates 
from Oct 12–Oct 17, 2026.

## AI/ML News Roundu...

(2) [0.85] Google, Meta, and OpenAI make major AI updates this week
https://www.facebook.com/groups/fdmahub/posts/2077326572915968
Google released a powerful new model. Gemini Skills started rolling out. ChatGPT Voice got an upgrade. Facebook is 
letting people talk to its algorithm....

(3) [0.83] AI Is About to Change Science Forever. And This Week Gave Us a Glimpse of How. | by Erick Galaviz | 
Automation for a Calmer Life | Oct, 2026 | Medium
https://medium.com/automation-for-a-calmer-life/ai-is-about-to-change-science-forever-and-this-week-gave-us-a-glimp
se-of-how-8a21522d76a1
Press enter or click to view image in full size

What if the most important AI breakthrough of 2026 has nothing to do with writing emails, generating images, or...

(4) [0.80] News from the week beginning 12th October 2026 -
https://www.enterprisetimes.co.uk/2026/10/19/news-from-the-week-beginning-12th-october-2026
# News from the week beginning 12th October 2026

By

Steve Brooks

-

FacebookXWhatsAppLinkedinEmail

fall autumn Image by Valentin Sabau from PixabaySeveral c...

(5) [0.78] AI News Today, October 19: Top Stories | AI Weekly
https://aiweekly.co/ai-news-today
Last 4 issues — click to explore

Agents

This week: 113 Anthropic

This week: 88 Chips

This week: 84 Regulation

This week: 73 OpenAI

This week: 65 AI Infras...

### Extract

Search returned ranked snippets. When a snippet isn't enough, we pass the URL(s) we care about to **Extract** and get the full page as clean markdown.

Same idea as Search: one client method, same knobs you'd pass over HTTP.

| Param | What it does |
|-------|--------------|
| `urls` | One URL or a list (max 20) |
| `extract_depth` | `basic` (faster) or `advanced` (better on JS-heavy pages) |
| `format` | `markdown` (default) or `text` |
| `query` | Optional focus string — reranks chunks toward what's relevant |
| `chunks_per_source` | With `query`, how many chunks per page (1–5) |

Below we extract the top result from the search we just ran — the classic search → extract pattern.


In [8]:
# Reuse a URL from the search response above.
url_to_extract = data["results"][0]["url"]
print(f"Extracting: {url_to_extract}")

extract_data = tavily_client.extract(
    urls=[url_to_extract],
    extract_depth="advanced",
    format="markdown",
)

print(f"Got {len(extract_data['results'])} page(s) in {extract_data.get('response_time')}s")
print(list(extract_data.keys()))


Extracting: https://medium.com/@techlatest.net/techlatest-ai-tech-weekly-38-3582a39f1738

Got 1 page(s) in 0.01s

['results', 'failed_results', 'response_time', 'request_id']

### What comes back from Extract

- **`results`** — one entry per successful URL, each with `url` and `raw_content` (the cleaned page text/markdown).
- **`failed_results`** — URLs that couldn't be fetched, with an error reason.
- **`response_time`**, **`request_id`** — metadata.

Compare `raw_content` to the short `content` snippet from Search — this is the full article the model can actually read.


In [9]:
for r in extract_data["results"]:
    content = r["raw_content"] or ""
    print(f"URL: {r['url']}")
    print(f"Length: {len(content)} chars")
    print(
        f"Preview:\n{content[:500]}\n\n[red]***[TRUNCATED FOR NOTEBOOK]***[/red]\n\n{content[-500:]}"
    )

if extract_data.get("failed_results"):
    print("Failed:", extract_data["failed_results"])


URL: https://medium.com/@techlatest.net/techlatest-ai-tech-weekly-38-3582a39f1738

Length: 21445 chars

Preview:
[Sitemap](https://medium.com/sitemap/sitemap.xml)

[Open in 
app](https://play.google.com/store/apps/details?id=com.medium.reader&referrer=utm_source%3DmobileNavBar&source=---t
op_nav_layout_nav-------------------------------------------)

Sign up

[Sign 
in](https://medium.com/m/signin?operation=login&redirect=https%3A%2F%2Fmedium.com%2F%40techlatest.net%2Ftechlatest-
ai-tech-weekly-38-3582a39f1738&source=post_page---top_nav_layout_nav-----------------------global_nav--------------
------)

[](https

***[TRUNCATED FOR NOTEBOOK]***

icy-f03bf92035c9?source=post_page-----3582a39f1738-----------------------------------------)

[Rules](https://policy.medium.com/medium-rules-30e5502c4eb4?source=post_page-----3582a39f1738----------------------
-------------------)

[Terms](https://policy.medium.com/medium-terms-of-service-9db0094a1e0f?source=post_page-----3582a39f1738-----------
------------------------------)

[Text to 
speech](https://speechify.com/medium?source=post_page-----3582a39f1738-----------------------------------------)

## 3 — A "tool" is just a function

An agent tool is nothing exotic — it's a normal Python function with a clear input and output. Here we wrap both Tavily endpoints:

- `internet_search()` — discover sources from a query
- `extract_content()` — read full pages from URLs we already have

We also add a small helper that flattens search results into a compact string we can hand back to the model.


In [10]:
def format_results(results: list[dict]) -> str:
    """Turn search results into a compact, model-friendly string."""
    if not results:
        return "No results found."
    return "\n\n".join(
        f"({i + 1}) Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}"
        for i, r in enumerate(results)
    )


def internet_search(
    query: str, search_depth: str = "advanced", max_results: int = 5
) -> str:
    """Search the web and return a formatted string of {title, url, content} results."""
    data = tavily_client.search(
        query, search_depth=search_depth, max_results=max_results
    )
    return format_results(data["results"])


def extract_content(urls: list[str], query: str | None = None) -> str:
    """Extract full page content from one or more URLs as markdown."""
    kwargs: dict = {
        "urls": urls,
        "extract_depth": "advanced",
        "format": "markdown",
    }
    if query:
        kwargs["query"] = query

    data = tavily_client.extract(**kwargs)
    results = data.get("results") or []
    if not results:
        failed = data.get("failed_results") or []
        return (
            f"No content extracted. Failures: {failed}"
            if failed
            else "No content extracted."
        )

    parts = []
    for i, r in enumerate(results):
        content = r.get("raw_content") or ""
        parts.append(f"({i + 1}) URL: {r['url']}\nContent:\n{content}")
    return "\n\n".join(parts)


print("=== Search ===")
print(internet_search("advancements in AI this week", max_results=3))

print("\n=== Extract ===")
sample_url = "https://www.allbirds.com"
print(f"(extracting {sample_url})\n")
print(extract_content([sample_url], query="product pricing")[:800])


=== Search ===

(1) Title: Huge AI Breakthroughs That Flew Under The Radar (And other AI News)
URL: https://www.youtube.com/watch?v=-Pk3DUNZxGI
Content: # Huge AI Breakthroughs That Flew Under The Radar (And other AI News)
## Channel: Matt Wolfe (verified)
1.01M subscribers
3.8K likes

### Description
100,394 views
Posted: 2025-01-17
Some major advances in AI this week. Here's what you missed... Don't forget to grab the free HubSpot prompt 
playbook here:  

Discover More:
🛠️ Explore AI Tools & News: 
📰 Weekly Newsletter: 
🎙️ The Next Wave Podcast:    / @thenextwavepod  

Socials:
🖼️ Instagram:   / mr.eflow  
❌ Twiter/X: 
🧵 Threads: 
🟦 LinkedIn:   / matt-wolfe-30841712 [...] [4:30] this week starting with Titans and well to be honest this one 
didn't actually come out this week it came out at the end of December so this one's a little
[4:38] over 2 weeks old now however it seems to be getting renewed attention this week especially after my buddy 
Matthew Burman did an excellent breakdown of this paper
[4:47] in a recent YouTube video This research comes out of Google and it argues for a new architecture called 
Titans as opposed to the traditional Transformer
[4:55] model that is what underlines pretty much all of anthropics and gpts and all of the normal existing large 
language models we're used to it says here we
[5:04] present a new neural long-term memory module that learns to memorize historical context and helps an 
attention to attend to the current [...] ### Transcript
[0:00] well this week feels like a week where there's been some legitimately big breakthroughs in the world of AI 
and I don't want to waste your time so let's Dive Right
[0:07] [Music]
[0:09] In now the first thing I want to share doesn't actually feel like one of those legitimately huge 
breakthroughs but it is one of the things that I feel like
[0:18] the most people are talking about so I'll start right here with Chad GPT now having a tasks feature inside 
of chat
[0:25] GPT you can now ask the AI assistant to schedule reminders on recurring requests now this is only available 
in the
[0:32] various paid plans and if we jump into chat GPT here this is how you'd find it up on the left where you 
select which model you're using if you're on one of

(2) Title: AI news October 2026: nine developments · Hello Growth
URL: https://hellogrowth.ai/en/blog/ai-news-october-2026
Content: There is AI news every week, but little of it changes anything for an organisation. From late September 
and the first week of October, nine developments stand out because they affect what you pay, who does the work and 
how far you can trust an AI system.

The common thread: AI is shifting from an assistant that answers questions to agents that carry out work on their 
own. That saves time, but only if you cap the costs, decide what an agent may do and make someone accountable. 
Below, for each development: what happened, what it means and where to read more. This overview was updated on 9 
October with GPT-6 in ChatGPT and Claude Haiku 5.5, and on 11 October with three developments at Google. All of 
this month's Google news is also in Google AI news October 2026. [...] > In short
>
>  AI news from late September and early October 2026 has one common thread: AI is moving from experiments to real 
work, and that calls for limits, permissions and an owner.
>  Nine developments matter for businesses: pay-as-you-go agents in Microsoft Copilot, Anthropic's investment in 
implementation talent, OpenAI's openness about unwanted model behaviour, Gemini Enterprise querying data without 
copying it, GPT-6 with interactive answers in ChatGPT, the cheaper Claude Haiku 5.5, Google's Gemini agent, skills 
replacing Gems, and a Mac app that takes meeting notes locally.

(3) Title: Artificial Intelligence News -- ScienceDaily
URL: https://www.sciencedaily.com/news/computers_math/artificial_intelligence
Content: ### Sunday, June 22, 2025

 Affordances in the Brain: The Human Superpower AI Hasn’t Mastered

##

=== Extract ===

(extracting https://www.allbirds.com)

(1) URL: https://www.allbirds.com
Content:
Free ground shipping on orders over $100

Due to increased demand, orders may take up to 30 days to ship.

## Added to Cart

Spend more to earn free shipping!

Shipping  $5.00

Subtotal

    

  All New Dasher NZ Collection

  # Wildly Comfortable. Super Natural.

  SHOP MEN SHOP WOMEN

   

  ## New Arrivals

    Shop Men   Shop Women
   

  ## Mens

    Shop Men
   

  ## Womens

    Shop Women
   

  ## Best Sellers

    Shop Men   Shop Women

NEWNEW

Women's Canvas Cruiser Slip On - Warm White (Natural White Sole) Women's Canvas Cruiser Slip On - Warm White 
(Natural White Sole)

Women's Canvas Cruiser Slip On

NEWNEW

Women's Cruiser Slip On - Blizzard (Blizzard Sole) Women's Cruiser Slip On - Blizzard (Blizzard Sole)

Women's Cruiser Slip On

## 4 — Let the model decide to call the tool

Up until now, we have only worked with the "tool" part of an agent.

Now the LLM. We point the `openai` SDK at Nebius's OpenAI-compatible endpoint, then **describe** our tools to the model as JSON schemas — both Search and Extract.

The crucial idea: the model never runs the functions itself. When it decides a tool is needed, it returns a **tool call** — the function name plus the arguments it wants us to run. Executing it is *our* job.


In [11]:
from openai import OpenAI

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=os.environ["NEBIUS_API_KEY"],
)

MODEL = "nvidia/nemotron-3-super-120b-a12b"

# The schema is how the model "sees" our function: name, purpose, and arguments.
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "internet_search",
        "description": "Search the web for current, factual information. Returns results with title, url, and a content snippet. Use this first to discover sources.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query."},
                "search_depth": {
                    "type": "string",
                    "enum": ["basic", "fast", "advanced"],
                    "description": "Search depth. Use 'advanced' for more detailed results.",
                },
                "max_results": {
                    "type": "integer",
                    "description": "How many results to return (1-10).",
                },
            },
            "required": ["query"],
        },
    },
}

EXTRACT_TOOL = {
    "type": "function",
    "function": {
        "name": "extract_content",
        "description": "Extract the full content of one or more web pages given their URLs. Use after search when a snippet is not enough and you need the full page.",
        "parameters": {
            "type": "object",
            "properties": {
                "urls": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "List of URLs to extract (max 5).",
                },
                "query": {
                    "type": "string",
                    "description": "Optional focus query to keep only the most relevant chunks from each page.",
                },
            },
            "required": ["urls"],
        },
    },
}

AGENT_TOOLS = [SEARCH_TOOL, EXTRACT_TOOL]


In [12]:
from datetime import datetime

messages = [
    {
        "role": "system",
        "content": (
            f"You are a research assistant. Use internet_search to find sources, "
            f"then extract_content on the most promising URLs when you need more detail. "
            f"Today is {datetime.now().strftime('%Y-%m-%d')}"
        ),
    },
    {"role": "user", "content": "What were the advancements in AI this week?"},
]

response = client.chat.completions.create(
    model=MODEL, messages=messages, tools=AGENT_TOOLS
)
assistant_msg = response.choices[0].message

print("Content (usually empty when it wants a tool):", repr(assistant_msg.content))
print("Tool calls the model is requesting:")
for tc in assistant_msg.tool_calls or []:
    print(f"  {tc.function.name}({tc.function.arguments})")


Content (usually empty when it wants a tool): None

Tool calls the model is requesting:

internet_search({"query": "AI advancements October 2026 week news breakthroughs", "search_depth": "advanced", 
"max_results": 10})

Notice the model didn't answer — it handed us a **request to act**: "please run this tool with these arguments."

Running it and returning the result is *our* job. Do that, feed the result back, and the model continues — it may search again, extract a page, or finally answer. Wrap that back-and-forth in a loop and you have a basic agent.


In [14]:
import json

# The model asked us to run a tool. Let's fulfil that request by hand, then hand
# the results back so it can finish -- ONE turn of what will become the loop.

TOOLS = {
    "internet_search": internet_search,
    "extract_content": extract_content,
}

# 1) Record the model's tool-call turn in the conversation history.
messages.append(assistant_msg.model_dump(exclude_none=True))

# 2) Run each tool call the model requested, appending each result as a "tool" message.
for tc in assistant_msg.tool_calls:
    args = json.loads(tc.function.arguments)
    print(f"Running {tc.function.name}({args})")
    result = TOOLS[tc.function.name](**args)
    print(f"Result snippet: {result[:100]} ... {result[-100:]}")
    messages.append(
        {
            "role": "tool",
            "tool_call_id": tc.id,
            "content": result,
        }
    )

# 3) Ask the model again -- now it can see the tool results and can answer (or call again).
followup = client.chat.completions.create(
    model=MODEL, messages=messages, tools=AGENT_TOOLS
)
final = followup.choices[0].message

if final.content:
    print("\nFinal answer:\n")
    print(final.content)
else:
    print("\nThe model wants another tool call:")
    for tc in final.tool_calls or []:
        print(f"  {tc.function.name}({tc.function.arguments})")
    print("...which is exactly why we wrap this in a loop next.")


Running internet_search({'query': 'AI advancements October 2026 week news breakthroughs', 'search_depth': 
'advanced', 'max_results': 10})

Result snippet: (1) Title: Episode 226: Weekly AI News Summary - 04 October 2026
URL: https://www.youtube.com/watch? ...  honorees check off all those boxes, in areas ranging from workplace 
productivity to medical safety.

The model wants another tool call:

extract_content({"urls": ["https://www.youtube.com/watch?v=M-z8e3eXUgs"], "query": "Weekly AI News Summary 
October 4 2026 advancements"})

...which is exactly why we wrap this in a loop next.

## 5 — The Pythonic agentic loop

The whole pattern is a `while` loop over a growing list of messages:

1. Ask the model, offering it both tools.
2. If it returns tool calls → run each one, append the results, loop again.
3. If it returns plain text → it's done; that text is the answer.

We add a `max_steps` guard so a confused model can't loop forever, and a few `print`s so we can watch each step. That's the entire idea behind "agents" — everything fancier is an optimization on top of this loop.


In [20]:
import json
from datetime import datetime
from typing import Callable

TOOLS: dict[str, Callable[..., str]] = {
    "internet_search": internet_search,
    "extract_content": extract_content,
}

SYSTEM_PROMPT = f"""You are a research assistant. Use internet_search to discover sources, then extract_content on the most promising URLs when snippets are not enough. Base your answer only on what you find, and cite the source URL for every claim. Search a maximum of 3 times, and use extract_content only once. Today is {datetime.now().strftime("%Y-%m-%d")}"""


def run_agent(
    question: str,
    model: str = MODEL,
    max_steps: int = 10,
    verbose: bool = True,
) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=model, messages=messages, tools=AGENT_TOOLS
        )
        msg = response.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:
            if verbose:
                print(f"[bold green]Step {step}: final answer[/bold green]")
            return msg.content

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose:
                print(f"[bold cyan]Step {step}: {tc.function.name}[/bold cyan] {args}")
            result = TOOLS[tc.function.name](**args)
            if verbose:
                print(f"Result snippet: {result[:100]} ... {result[-100:]}")
            messages.append(
                {"role": "tool", "tool_call_id": tc.id, "content": result}
            )

    return "Stopped: reached max_steps without a final answer."


In [19]:
answer = run_agent(
    "What is the TipiCalc?"
    "Search first, then extract the full text of the 1-2 most useful articles before answering.",
)

print("[bold]Answer[/bold]\n")
print(answer)


Step 1: extract_content {'urls': ['https://www.youtube.com/watch?v=M-z8e3eXUgs'], 'query': 'Weekly AI News Summary 
October 4 2026 advancements'}

Result snippet: (1) URL: https://www.youtube.com/watch?v=M-z8e3eXUgs
Content:
In today’s Weekly AI News Summary, we  ... rarching theme here is the total removal of friction. When a machine 
translates your voice with zero

Step 2: extract_content {'urls': ['https://www.youtube.com/watch?v=M-z8e3eXUgs'], 'query': 'Weekly AI News Summary 
October 4 2026 advancements'}

Result snippet: (1) URL: https://www.youtube.com/watch?v=M-z8e3eXUgs
Content:
In today’s Weekly AI News Summary, we  ... rarching theme here is the total removal of friction. When a machine 
translates your voice with zero

Step 3: extract_content {'urls': ['https://www.youtube.com/watch?v=M-z8e3eXUgs'], 'query': 'Weekly AI News Summary 
October 4 2026 advancements'}

...

Step 4: extract_content {'urls': ['https://www.youtube.com/watch?v=M-z8e3eXUgs'], 'query': 'Weekly AI News Summary 
October 4 2026 advancements'}

Result snippet: (1) URL: https://www.youtube.com/watch?v=M-z8e3eXUgs
Content:
In today’s Weekly AI News Summary, we  ... rarching theme here is the total removal of friction. When a machine 
translates your voice with zero

Step 5: final answer

Answer

I was unable to locate any reliable information about “TipiCalc” in the available search results. The queries 
returned only unrelated content (a YouTube video about AI news), and no pages describing what TipiCalc is, its 
purpose, features, or usage were found. Consequently, I cannot provide a description of TipiCalc based on 
verifiable sources.

## 6 — Bonus: how good are different models at tool calling?

Not every model is equally reliable at *deciding* to call a tool, formatting *valid* arguments, and *stopping* when done. Because our loop is model-agnostic, we can point it at several Nebius models and eyeball the difference. Swap in any model IDs from the [Nebius playground](https://tokenfactory.nebius.com); bad names are caught by the `try/except` below.

In [ ]:
CANDIDATE_MODELS = [
    "nvidia/nemotron-3-super-120b-a12b",
    "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B",
    "MiniMaxAI/MiniMax-M2.5",
]

question = "What is the latest stable version of Python, and when was it released?"

for model in CANDIDATE_MODELS:
    print(f"[bold]{model}[/bold]")
    try:
        answer = run_agent(
            question, model=model, max_steps=5, verbose=False
        )
        print(answer[:400])
    except Exception as e:
        print(f"[red]failed: {type(e).__name__}: {e}[/red]")
    print("-" * 80)


## 7 — Where this breaks, and what's next

Our 20-line agent works, but it's naive:

- **No planning.** It reacts one search/extract at a time; it doesn't decompose a big question into sub-tasks.
- **Redundant calls.** Nothing stops it re-querying the same thing or re-extracting the same URL.
- **No source-quality control.** It trusts whatever ranks highest.
- **Context bloat.** Every result (especially full-page extracts) is stuffed back into the message list; long runs blow past the context window.

Fixing these is the job of a real **search pipeline** — query planning, result deduplication, source filtering, and summarization — which we build in the next chapter. From there we graduate to a full multi-agent **competitor-research agent** (see the project code in the repo root) that plans, delegates to parallel sub-agents, and fact-checks its own output.


## 8 — Practice: own the research pattern

The demo used one running example so the steps would connect. Practice is where you **transfer** that pattern to new questions — same loop, different research jobs.

Keep `verbose=True`. Don't count tool calls; watch *which* tool fires and whether the answer actually needed full pages or just snippets.


### Exercise 1 — Match the job to the tool

For each research job below, decide **before running**: Search only, Extract only, or Search → Extract. One line of reasoning is enough ("I don't know the URL yet" / "snippets won't cover pricing tiers" / …).

| Job | Hint |
|-----|------|
| A. Who is leading the F1 constructors' championship right now? | Fresh fact; a good snippet often wins. |
| B. You already have `https://www.allbirds.com` — list what's on the product homepage in enough detail to brief a teammate. | URL is known; you need page text. |
| C. Compare Cursor pricing with Devin pricing *from the vendors' own pages*. | Discover URLs, then read them in full. |

Then run **one** of them with the agent and check whether your call matched what the model actually did. If it disagreed, was the model wrong — or was your assumption about snippet depth wrong?


In [ ]:
# Your pick: A / B / C (or your own). Note Search / Extract / both above, then run.
YOUR_QUESTION = (
    "Who is leading the F1 constructors' championship right now?"
)

# For job B you could instead ask something like:
# YOUR_QUESTION = (
#     "Extract https://tavily.com and brief me on what the product does "
#     "and who it's for. Use extract_content on that URL."
# )

answer = run_agent(YOUR_QUESTION, verbose=True)
print("[bold]Answer[/bold]\n")
print(answer)


### Exercise 2 — Same topic, different depth

Run the agent twice on **Nebius / Token Factory** (or another product you care about):

1. A *headline* question — what shipped or got announced recently?
2. A *primary-source* question — find the announcement or docs page and pull specifics (pricing, model names, limits) from the page itself.

Only change the user question. Compare the verbose traces: did wording alone push the model from Search-only into Search → Extract? If it still skipped Extract, what in the system prompt or the snippets might explain that?


In [ ]:
HEADLINE = (
    "What has Nebius Token Factory announced or launched recently?"
)

PRIMARY_SOURCE = (
    "Find Nebius Token Factory's pricing or models page, extract the full page, "
    "and list 3 concrete details (model names, limits, or price signals) with URLs."
)

print("[bold]Run 1 — headline[/bold]")
print(run_agent(HEADLINE, verbose=True))
print("\n" + "=" * 80 + "\n")
print("[bold]Run 2 — primary source[/bold]")
print(run_agent(PRIMARY_SOURCE, verbose=True))


### Exercise 3 — Break it on purpose (concept check)

No need to write code unless you want to. Answer briefly:

1. **Decide vs execute.** If we deleted the `for tc in msg.tool_calls:` block but still passed `tools=AGENT_TOOLS` to the API, what would the model do — and what would the user see?
2. **Why append?** Suppose we ran the tool but *forgot* to append the `role: "tool"` message. What goes wrong on the next `chat.completions.create` call?
3. **Search vs Extract.** You're building the competitor-research agent this course is heading toward. A sub-task is "pull the competitor's pricing page and list every plan." Which tool is the workhorse?

<details>
<summary>Spoilers / sample answers</summary>

1. The model can still *request* tool calls, but nothing runs them. The loop would either stall on empty content + pending tool_calls, or you'd return nothing useful — the host never fulfilled the contract.
2. The API expects a tool result for each requested `tool_call_id`. Missing them usually errors or leaves the model without observations, so it can't honestly continue.
3. **Extract** is the workhorse once you have the URL (full plan tables rarely fit in snippets). **Search** is how you *find* the right pricing URL when you don't already know it.

</details>
